# 01 — Kiểm tra chất lượng AI4I

Chạy từ kernel mới trong môi trường dự án. Cần tải CSV theo `datasets/README.md`; checksum phải khớp manifest. Không sửa nguồn, không tạo split hoặc học mô hình.

Phạm vi: kiểm tra cấu trúc/nhãn trên toàn nguồn. Nhãn chỉ dùng kiểm tra tính nhất quán và thống kê mô tả; không chọn feature, ngưỡng hoặc tham số dựa trên kết quả phân biệt nhãn ở đây. Các kiểm tra này phải được công khai khi mô tả giao thức đánh giá sau này.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "configs/ai4i-schema.json").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Mở notebook bên trong repository dự án.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from maintenance.quality import analyze, export_analysis, plot_analysis
result = analyze(ROOT)


In [2]:
result["summary"]

{'rows': 10000,
 'columns': 14,
 'dataset_version': 'sha256:dc6630cd9b1f0f853922fad78a1b6436570d3f1ec863f1dd5c4340ac56bc8a8e',
 'schema_version': 'ai4i-v1',
 'missing_cells': 0,
 'duplicate_reading_id': 0,
 'duplicate_full_row': 0,
 'duplicate_feature_rows': 0,
 'target_positive': 339,
 'target_rate': 0.0339,
 'failure_without_mode': 9,
 'mode_without_failure': 18,
 'multiple_modes': 24,
 'scope': 'full-source descriptive audit; no split or model selection',
 'source_sha256': 'dc6630cd9b1f0f853922fad78a1b6436570d3f1ec863f1dd5c4340ac56bc8a8e'}

## Thiếu, sai kiểu, sai miền và tính duy nhất
Không tự điền thiếu, sửa nhãn hoặc loại ngoại lệ.

In [3]:
display(result["column_audit"])
display(result["labels"])
display(result["by_type"])

,column,role,missing,invalid_type,invalid_domain,unique_nonmissing
0,reading_id,identifier,0,0,0,10000
1,product_id,identifier,0,0,0,10000
2,product_type,feature,0,0,0,3
3,air_temperature_k,feature,0,0,0,93
4,process_temperature_k,feature,0,0,0,82
5,rotational_speed_rpm,feature,0,0,0,941
6,torque_nm,feature,0,0,0,577
7,tool_wear_min,feature,0,0,0,246
8,machine_failure,target,0,0,0,2
9,twf,failure_mode,0,0,0,2


,positive_count,rate
machine_failure,339,0.0339
twf,46,0.0046
hdf,115,0.0115
pwf,95,0.0095
osf,98,0.0098
rnf,19,0.0019


,records,failures,failure_rate
product_type,,,
H,1003,21,0.020937
L,6000,235,0.039167
M,2997,83,0.027694


## Nhãn tổng và nhãn chế độ

`failure_without_mode` và `mode_without_failure` là hai dạng bất nhất. `multiple_modes` là nhãn chồng lấp, không tự coi là lỗi. Không công bố ID cấp bản ghi trong notebook; danh sách cờ được xuất local để truy vết.

In [4]:
export_analysis(result, ROOT)
print("Đã xuất bảng tổng hợp vào reports/tables/quality; cờ cấp bản ghi vào artifacts/metrics/quality local.")

Đã xuất bảng tổng hợp vào reports/tables/quality; cờ cấp bản ghi vào artifacts/metrics/quality local.


## Tiếp theo

Xem `docs/DATA_QUALITY.md` để đọc bằng chứng và quyết định cần theo dõi. Không cộng số cờ thành số bản ghi bị lỗi vì các nhóm có thể giao nhau. Notebook này chưa đánh giá mô hình.